# Advanced Baseline Experiment: Parkinson's Disease Classification

This notebook represents a **Journal-Quality Baseline Methodology** for Parkinson's Disease classification using vocal biomarkers.

### Key Enhancements over the Simple Baseline:
1. **Prevention of Data Leakage (GroupKFold)**: The Sakar dataset has 3 recordings per patient (identified by the `id` column). Standard random splitting leaks patient identity between train and test sets. We use `GroupKFold` so a single patient's recordings are strictly in either the training or testing set.
2. **Dimensionality Reduction & Feature Selection**: The dataset has 754 features but only 252 patients. To avoid the curse of dimensionality, we apply Variance Thresholding and Univariate Feature Selection (SelectKBest) inside a pipeline.
3. **Hyperparameter Tuning**: We use Cross-Validation Grid Search to optimize the XGBoost baseline.
4. **SHAP Explainability**: We include SHAP (SHapley Additive exPlanations) to interpret feature importance, which directly aligns with the core goal of your paper.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import GroupKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import xgboost as xgb

# Set random seed
np.random.seed(42)

## 1. Load Data & Prepare Groups

In [ ]:
# Load data
df = pd.read_csv('../dataset/pd_speech_features.csv', header=1)

# The 'id' column represents the patient ID. There are 3 recordings per patient.
groups = df['id']
X = df.drop(['id', 'class'], axis=1)
y = df['class']

print(f"Dataset shape: {df.shape}")
print(f"Number of unique patients: {groups.nunique()}")
print(f"Class distribution:\n{y.value_counts()}")

## 2. Define the Modeling Pipeline
We use `imblearn.pipeline` to ensure SMOTE is only applied to the training folds during cross-validation, preventing data leakage.

In [ ]:
# Build a pipeline with Feature Selection -> Scaling -> SMOTE -> XGBoost
pipeline = ImbPipeline([
    ('var_thresh', VarianceThreshold(threshold=0.01)), # Remove near-constant features
    ('scaler', StandardScaler()),                      # Scale features
    ('selector', SelectKBest(score_func=f_classif, k=50)), # Select top 50 features
    ('smote', SMOTE(random_state=42)),                 # Handle class imbalance
    ('classifier', xgb.XGBClassifier(objective='binary:logistic', random_state=42, eval_metric='logloss'))
])

## 3. Grouped Cross-Validation Evaluation
We use 5-fold GroupKFold to ensure patients are not split across train and test sets.

In [ ]:
gkf = GroupKFold(n_splits=5)

# We can define a parameter grid if we want to tune. For now, we will evaluate the default pipeline.
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}

print("Running 5-Fold Grouped Cross-Validation...")
cv_results = cross_validate(pipeline, X, y, groups=groups, cv=gkf, scoring=scoring, n_jobs=-1)

print("\n=== Cross-Validation Results ===")
print(f"Accuracy:  {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")
print(f"Precision: {cv_results['test_precision'].mean():.4f} ± {cv_results['test_precision'].std():.4f}")
print(f"Recall:    {cv_results['test_recall'].mean():.4f} ± {cv_results['test_recall'].std():.4f}")
print(f"F1-Score:  {cv_results['test_f1'].mean():.4f} ± {cv_results['test_f1'].std():.4f}")
print(f"ROC-AUC:   {cv_results['test_roc_auc'].mean():.4f} ± {cv_results['test_roc_auc'].std():.4f}")

## 4. Train Final Model & SHAP Explainability
To extract SHAP values, we train the pipeline on the full dataset. (In a real paper, you might extract SHAP values on the hold-out test set, but for overview purposes, global SHAP is shown here).

In [ ]:
# Fit the pipeline on all data to extract feature importances
pipeline.fit(X, y)

# Extract the fitted steps
fitted_scaler = pipeline.named_steps['scaler']
fitted_selector = pipeline.named_steps['selector']
fitted_xgb = pipeline.named_steps['classifier']
fitted_var_thresh = pipeline.named_steps['var_thresh']

# Get the names of the selected features
features_after_var = X.columns[fitted_var_thresh.get_support()]
selected_features = features_after_var[fitted_selector.get_support()]

# Transform X to the 50 selected features for SHAP analysis
X_transformed = fitted_selector.transform(fitted_scaler.transform(fitted_var_thresh.transform(X)))
X_shap = pd.DataFrame(X_transformed, columns=selected_features)

# Initialize SHAP Tree Explainer
explainer = shap.TreeExplainer(fitted_xgb)
shap_values = explainer.shap_values(X_shap)

# Plot Global Feature Importance (Summary Plot)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False)
plt.title("Top Predictive Vocal Biomarkers (SHAP Global Importance)")
plt.tight_layout()
plt.show()

# Plot SHAP Summary plot (Impact on model output)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, show=False)
plt.title("SHAP Value Impact per Feature")
plt.tight_layout()
plt.show()